In [ ]:
import sys
from pathlib import Path

SSS_ROOT = Path.cwd().parent      # .../SSS
MORAI_ROOT = SSS_ROOT.parent      # .../MORAI
sys.path.append(str(SSS_ROOT))    # for `inference.*`
sys.path.append(str(MORAI_ROOT))  # for `SSS.fovea` (imported inside inference/qwen_vdgd.py)

from inference.qwen_base import qwen_base, load_qwen
from inference.qwen_vdgd import qwen_vdgd

import os
import io
import base64
import pandas as pd
from PIL import Image
from IPython.display import display

os.environ["CUDA_VISIBLE_DEVICES"] = '2'

# Qwen2.5-VL image-resolution bounds (multiples of the 28x28 patch size);
# applied identically to base and vdgd so the comparison isolates the
# decoding-time difference and not image-resolution differences.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1280 * 28 * 28

Load Mmerror

In [ ]:
# image_path = 
# question_path = 
# image = Image.open(image_path).convert("RGB")
# with open(question_path, "r", encoding="utf-8") as f:
#     data = json.load(f)
# question = data["question"]
# answer = data["correct_answer"]

# display(image)
# print(question)
# print(f'Answer: {answer}')

Load TreeBench

In [ ]:
tsv_path = SSS_ROOT / "data/TreeBench/TreeBench.tsv"
df = pd.read_csv(tsv_path, sep="\t")

row = df.iloc[0]  # pick a row; swap the index to try others
image = Image.open(io.BytesIO(base64.b64decode(row["image"]))).convert("RGB")
display(image)

question = row["question"]
if isinstance(row["multi-choice options"], str) and row["multi-choice options"]:
    # matches eval_treebench.py's " Options:\n" separator
    question = question + " Options:\n" + row["multi-choice options"]

ground_truth = str(row["answer"]).strip().upper()
print(question)
print(f"Ground truth: {ground_truth}")

In [ ]:
# Load once, share between the base and VDGD runs so the comparison isolates
# the decoding-time difference (logits_processor) and nothing else.
model, processor = load_qwen(str(MORAI_ROOT / "weights/Qwen2.5-VL-7B-Instruct"))

In [ ]:
# --- base: greedy decoding, no logits processor ---
base_answer = qwen_base(image, question, model=model, processor=processor, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
print(base_answer)

In [ ]:
# --- vdgd: describe -> prefix -> prime scorer -> decode with VDGDLogitsProcessor ---
vdgd_answer, description = qwen_vdgd(
    image, question, model=model, processor=processor, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS
)
print(description)
print("--------")
print(vdgd_answer)